Set Up TensorFLow

In [1]:
import tensorflow as tf
print("TensorFlow version:", tf.__version__)

2026-02-01 01:54:06.370741: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-02-01 01:54:06.380507: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-01 01:54:06.715848: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-01 01:54:08.252619: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To tur

TensorFlow version: 2.20.0


LOAD A DATASET:

Load and prepare the MNIST dataset. The pixel values of the images range from 0 to 255. Scale these values to a range of 0 to 1 by dividing the values by 255.0 .This also converts the sample data from integers to floating numbers.

In [3]:
mnist = tf.keras.datasets.mnist
(x_train, y_train), (x_test, y_test) = mnist.load_data()
x_train, x_test = x_train/255.0, x_test/255.0

BUILDING A MACHINE LEARNING MODEL

Build a tf.keras.Sequential model

In [4]:
model = tf.keras.models.Sequential([
    tf.keras.layers.Flatten(input_shape=(28, 28)),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(18)
])

/home/izy/Documents/Project/TensorflowPractice/venv/lib/python3.12/site-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
2026-02-01 02:05:36.716499: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


SEQUENTIAL: is useful for stacking layers where each layer has one imput tensor one output tensor. Layers are functions with a known mathematical structure that can be reused and have a trainable variables. Most TensorFLow Models are composed of layers. This model uses the Flatten, Dense, and Dropout Layers.

For each example, the model returns a vetor of logits or log-odds scores, one for eah class.

In [5]:
predictions = model(x_train[:1]).numpy()
predictions

array([[ 0.27824163, -0.24951364, -0.25836012,  0.03242617, -0.6014404 ,
        -0.46709907,  0.27218318,  0.07997306,  0.42300615,  0.4193837 ,
         0.09000383,  0.39363316, -0.34107125,  0.2738923 ,  0.41819558,
        -0.65039235,  0.30202693,  0.23099524]], dtype=float32)

The tf.nn.softmax function converts these logits to probabilities for each class:

In [6]:
tf.nn.softmax(predictions).numpy()

array([[0.0667752 , 0.03939254, 0.03904559, 0.05222265, 0.02770602,
        0.03168967, 0.06637186, 0.05476565, 0.07717659, 0.07689753,
        0.05531776, 0.07494266, 0.03594603, 0.0664854 , 0.07680622,
        0.02638242, 0.0683825 , 0.06369367]], dtype=float32)

Defining  a loss function for training using losses.SparseCategoricalCrossentropy

In [8]:
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits
=True)

The loss function takes a vector of ground truth values and a vector of logits and returns a scalar loss for each example. This loss is equal to the negative log probability of the true class: The loss is zero if the model is sure of the correct class.

This untrained model gives probabilities close to random(1/10 for each class), so the initial loss should be close to -tf.math.log(1/10) ~=2.3

In [9]:
loss_fn(y_train[:1], predictions).numpy()

np.float32(3.4517643)

Before you start training, configure and compile the model using keras Model.compile. Set the optimizer class to adam, set the loss to the loss_fn function defined earlier, and specify a metric to be evaluated for the model by setting the metrics parameter to accuracy 

In [10]:
model.compile(optimizer='adam',
loss=loss_fn,
metrics=['accuracy'])

TRAIN AND EVALUATE MODEL

use the model.fit method to adjust your model parameters and minimize the loss

In [11]:
model.fit(x_train, y_train, epochs=5)

Epoch 1/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9115 - loss: 0.3040
Epoch 2/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9567 - loss: 0.1478
Epoch 3/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9667 - loss: 0.1104
Epoch 4/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9717 - loss: 0.0906
Epoch 5/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9756 - loss: 0.0776


The model.evaluate method checks the model's performance, usually on a validation set or test set

In [12]:
model.evaluate(x_test, y_test, verbose=2)

313/313 - 0s - 1ms/step - accuracy: 0.9778 - loss: 0.0743


[0.07428088784217834, 0.9778000116348267]

The image classifier is now trained to ~98% accuracy on this dataset. 

If you need to return the probality, you can wrap the trained model, and attach the softmax to it.

In [13]:
probability_model = tf.keras.Sequential([
    model,
    tf.keras.layers.Softmax()
])

In [14]:
probability_model(x_test[:5])

<tf.Tensor: shape=(5, 18), dtype=float32, numpy=
array([[4.05752987e-09, 8.23704216e-09, 3.21472862e-06, 8.26746982e-05,
        4.09792816e-12, 1.28125532e-06, 1.30850747e-13, 9.99904156e-01,
        4.95985432e-07, 8.10621168e-06, 2.86867795e-12, 3.30766491e-12,
        5.48417485e-13, 4.82210355e-13, 3.70415965e-13, 5.25809257e-12,
        2.74621775e-12, 2.81796700e-12],
       [3.07904520e-06, 6.29638438e-04, 9.99249995e-01, 2.08588644e-05,
        3.22192186e-12, 4.16722978e-05, 4.29665022e-07, 2.75326962e-13,
        5.42121816e-05, 1.56066150e-11, 1.14040152e-13, 1.59840145e-13,
        5.38463859e-13, 1.00496651e-12, 1.14925007e-12, 1.39354911e-14,
        3.71357459e-13, 4.34852024e-13],
       [1.48217453e-06, 9.98308301e-01, 5.06333527e-05, 1.17506734e-05,
        1.38410222e-04, 4.35110451e-06, 4.24576829e-05, 1.12908648e-03,
        3.08389252e-04, 4.92705476e-06, 1.22199593e-08, 9.45139522e-09,
        5.05057152e-08, 1.38484282e-08, 5.57310926e-08, 8.01086664e-09,
     